# Does the host's bias do anything?

An investigation of the biased host in `monty.py` (3 doors). The biased host knows where the prize is. When he has two doors he could open, he opens the leftmost (lowest-numbered) one with probability `lowerDoorProbability` (`--bias`).

Run this notebook from the project folder so that `import monty` works.

In [1]:
import numpy as np

from monty import makeBiasedHost, playGameWithPolicies, switchStrategy

SEED = 2026
NUM_GAMES = 100_000
LOWER_DOOR_PROBABILITIES = [0.1, 0.5, 0.9, 1.0]


def playSwitchGames(lowerDoorProbability):
    """Play NUM_GAMES 3-door switch games against the biased host and return the GameRecords."""
    rng = np.random.default_rng(SEED)
    biasedHost = makeBiasedHost(lowerDoorProbability)
    return [playGameWithPolicies(biasedHost, switchStrategy, rng) for _ in range(NUM_GAMES)]


gameRecordsByProbability = {lowerDoorProbability: playSwitchGames(lowerDoorProbability)
                            for lowerDoorProbability in LOWER_DOOR_PROBABILITIES}

## Round 1: does the bias change the overall win rate?

**Question.** After adding the biased host, does `--bias` change the switch win rate that the program prints?

**Method.** Play the same number of switch games at several bias values and compare the overall win rate.

In [2]:
for lowerDoorProbability, gameRecords in gameRecordsByProbability.items():
    switchWins = sum(gameRecord.won for gameRecord in gameRecords)
    print(f"bias {lowerDoorProbability:.1f}: switch wins {switchWins}/{NUM_GAMES} "
          f"({switchWins / NUM_GAMES:.4f})")

bias 0.1: switch wins 66627/100000 (0.6663)
bias 0.5: switch wins 66627/100000 (0.6663)
bias 0.9: switch wins 66627/100000 (0.6663)
bias 1.0: switch wins 66627/100000 (0.6663)


**Result.** The overall switch win rate stays at about 2/3 for every bias value. Implementing the bias did not change the program's output at all.

**What this leads to.** The program only reports the win rate over all games. A player at the doors sees *which* door the host opens, and the output never splits the games by that. So the next round has to look at the win rate after seeing the host's door.

## Round 2: win rate after seeing which door the host opens

**Question.** If the player knows the host is biased toward the leftmost door, does the door he actually opens tell the player anything?

**Prediction (made before running this).** If the host is biased toward the leftmost door and he opens a door that is *not* the leftmost, then the leftmost door is likely the one with the prize, so the player should switch to it. Switching should win much more often in that case, even though the program's output is the same for every bias.

**What would prove me wrong.** The switch win rate being about 2/3 in both groups.

**Method.** Reuse the games from round 1. For each game, look at the two doors the player did not choose and check whether the host opened the leftmost of those two or the other one. Compute the switch win rate separately for each group.

In [3]:
def splitByHostDoor(gameRecords):
    """Return (wonListWhenHostOpenedLeftmost, wonListWhenHostOpenedOther)."""
    wonWhenHostOpenedLeftmost = []
    wonWhenHostOpenedOther = []
    for gameRecord in gameRecords:
        unchosenDoors = [door for door in range(3) if door != gameRecord.firstChoice]
        if gameRecord.openedDoor == min(unchosenDoors):
            wonWhenHostOpenedLeftmost.append(gameRecord.won)
        else:
            wonWhenHostOpenedOther.append(gameRecord.won)
    return wonWhenHostOpenedLeftmost, wonWhenHostOpenedOther


print("bias | host opened leftmost door   | host opened the other door")
print("     | games    switch win rate   | games    switch win rate")
for lowerDoorProbability, gameRecords in gameRecordsByProbability.items():
    wonWhenHostOpenedLeftmost, wonWhenHostOpenedOther = splitByHostDoor(gameRecords)
    leftmostWinRate = sum(wonWhenHostOpenedLeftmost) / len(wonWhenHostOpenedLeftmost)
    otherWinRate = sum(wonWhenHostOpenedOther) / len(wonWhenHostOpenedOther)
    print(f" {lowerDoorProbability:.1f} | {len(wonWhenHostOpenedLeftmost):>6}   {leftmostWinRate:.4f}"
          f"            | {len(wonWhenHostOpenedOther):>6}   {otherWinRate:.4f}")

bias | host opened leftmost door   | host opened the other door
     | games    switch win rate   | games    switch win rate
 0.1 |  36864   0.9091            |  63136   0.5245
 0.5 |  50324   0.6660            |  49676   0.6666
 0.9 |  63476   0.5280            |  36524   0.9066
 1.0 |  66887   0.5011            |  33113   1.0000


**Result.** The prediction was right. With bias 0.9, when the host opens the door that is not the leftmost, switching wins about 91% of the time. When he opens the leftmost door as expected, switching wins only about 53% of the time. With bias 1.0 the two cases are 100% and about 50%. With bias 0.5 (no bias) both groups are about 2/3, so the host's door tells the player nothing. Bias 0.1 is the mirror image of 0.9: there the surprising door is the leftmost one.

**What I understand now.**

- The bias does do something. It changes how much the player learns from the door the host opens. It does not show in the program's output because the output averages over both groups, and the two groups average back to 2/3 (for bias 0.9: 63476 games at 0.5280 and 36524 games at 0.9066).
- The overall win counts in round 1 are exactly equal, not just close. With 3 doors a switch wins exactly when the first choice was wrong, and that does not depend on which door the host opens.
- Switching never drops below about 1/2 in any group, so "always switch" is still never a worse choice than sticking. What the bias changes is how sure the player can be.

**Limits of this investigation.** 3 doors only, one seed (2026), 100,000 games per bias value. The groups are smaller than 100,000, so their rates have more error than the overall rate.